# Q-Synth API Tutorial

This tutorial demonstrates the main features and usage of the Q-Synth API for optimal quantum circuit synthesis. 

We assume you have already installed the Q-Synth package. If not, Q-Synth can be installed using the following pip-command. You probably want to do that in your own virtual environment (with venv):

```
pip install --pre Q-Synth
```
---

## Basic Input, Output, Arguments

### Quantum Circuit
Loading a quantum circuit from a file in OPENQASM 2.0 format:

In [ ]:
from qiskit import QuantumCircuit

ecai24 = QuantumCircuit.from_qasm_file('ecai24.qasm')
print(ecai24)

Creating a quantum circuit using Qiskit:

In [ ]:
from qiskit import QuantumCircuit
def qc_example():
    qc = QuantumCircuit(3)
    qc.cx(0, 1)
    qc.s(0)
    qc.cx(0, 2)
    qc.cx(1, 2)
    return qc
print(qc_example())

### Specifying a Quantum Platform

For layout mapping and layout-aware synthesis, the coupling graph of the platform needs to be specified. It will be bi-directional by default. There are three methods:

1. Specifying the coupling graph explicitly:

In [ ]:
from qsynth import make_bidirectional_graph

# Custom coupling graph
coupling_graph = [[0,1], [1,2]]
# Make the coupling graph bidirectional
coupling_graph = make_bidirectional_graph(coupling_graph)
print(coupling_graph)

2. Specifying a predefined platforms (e.g., 'tenerife', 'melbourne', 'sycamore', 'rigetti-80', 'eagle')

In [ ]:
from qsynth import get_coupling_graph
# Predefined platform example (e.g., 'tenerife')
tenerife = get_coupling_graph('tenerife')
print(tenerife)

3. Generating an instance of a scalable platform (e.g. line-N, cycle-N, grid-N)

In [ ]:
# Scalable platform example (e.g., 'line-4' for nearest neighbor line of 4 qubits)
line4 = get_coupling_graph('line-4')
print(line4)

### Understanding Q-Synth output: MappedResult
The output of Q-Synth synthesis functions is a `MappedResult` object containing:
- `circuit`: the mapped/optimized quantum circuit
- `initial_mapping`: the initial mapping of qubits
- `final_mapping`: the final mapping of qubits after synthesis

In [ ]:
from qsynth import layout_synthesis
qc = qc_example()
result = layout_synthesis(circuit=qc, coupling_graph=line4, verbose=-1)
print('Mapped circuit:')
print(result.circuit)
print('Initial mapping:', result.initial_mapping)
print('Final mapping:', result.final_mapping)

## Layout Synthesis

### Minimal example:

In [ ]:
# Minimal layout synthesis example
from qsynth import layout_synthesis, get_coupling_graph
qc = qc_example()
coupling_graph = get_coupling_graph("line-3")
result = layout_synthesis(circuit=qc, coupling_graph=coupling_graph, metric='cx-count')
print('Mapped circuit:')
print(result.circuit)
print('Initial mapping:', result.initial_mapping)
print('Final mapping:', result.final_mapping)

### Using subarchitectures for large platforms:

When mapping to large platforms, we can use subarchitectures. By default, Q-Synth maps to all maximal subarchitectures with 0 ancillas. Note that we write the resulting circuit to a file.


In [ ]:
from qsynth import layout_synthesis, get_coupling_graph
from qiskit import QuantumCircuit, qasm2

barenco = QuantumCircuit.from_qasm_file('barenco_tof_3.qasm')
sycamore = get_coupling_graph('sycamore')
result = layout_synthesis(circuit=barenco, coupling_graph=sycamore, metric='cx-count', subarchitecture=True, verbose=0)
qasm2.dump(result.circuit, "mapped_barenco_tof_3.qasm")

We can also specify the number of ancillas to use with subarchitectures:

In [ ]:
result = layout_synthesis(circuit=barenco, coupling_graph=sycamore, metric='cx-count', subarchitecture=True, verbose=0,
                          num_ancillary_qubits=1)

### Optimization metrics

Q-Synth supports layout synthesis that is optimal for the number of CNOT gates (`cx-count`),
the depth of CNOT gates (`cx-depth`) or the total circuit depth (`depth`).
These can be combined with a *secondary* metric. For instance we can first find a depth-optimal mapping, and then search
for the lowest possible count without increasing the depth:

In [ ]:
clifford_circuit = QuantumCircuit.from_qasm_file('04q_33936_clifford.qasm')
line4 = get_coupling_graph("line-4")
result = layout_synthesis(clifford_circuit, line4, metric="cx-count", secondary_metric="cx-depth")

### Optimal layout synthesis for a given initial mapping

You can provide Q-Synth with an initial mapping, for instance one computed using Qiskit's SABRE heuristics. 
Q-Synth will then compute the minimally mapped layout for that initial mapping.

In [ ]:
# Example: Using Qiskit for initial mapping, then Q-Synth for optimal synthesis from that mapping
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import SabreLayout
vqe = QuantumCircuit.from_qasm_file('vqe_8_3_5_100.qasm')
sycamore = get_coupling_graph('sycamore')
layout_pass = SabreLayout(CouplingMap(sycamore), seed=1)
mapped_vqe = PassManager(layout_pass).run(vqe)
print("Qiskit SabreLayout mapped circuit counts:", mapped_vqe.count_ops())
initial_layout = dict(enumerate(mapped_vqe.layout.initial_index_layout(filter_ancillas=True)))
print('Initial layout from Qiskit SabreLayout:', initial_layout)
result = layout_synthesis(circuit=vqe, coupling_graph=sycamore, metric='cx-count', initial_mapping=initial_layout,
                          verbose=0)
print("Q-Synth mapped circuit counts:", result.circuit.count_ops())
print('Initial Mapping:', result.initial_mapping)
print('Final mapping:', result.final_mapping)


Using Q-Synth for the initial mapping computed by SabreLayout gives reduces the number of swaps by 1 (optimal for this initial mapping)

## CNOT Synthesis

CNOT synthesis applies to pure CNOT circuits, but can be applied to CNOT slices as well (see Peephole synthesis below).
Here we optimize for `cx-count`, but other metrics are supported as well.

We now explain the four basic variants: S, W, S+R, W+R. Note that
We will illustrate them on this example from the ECAI 2024-paper:

In [ ]:
from qiskit import QuantumCircuit
qc = QuantumCircuit.from_qasm_file('ecai24.qasm')
print('Original circuit:')
print(qc)

### Pure CNOT synthesis

1. Minimal example (S): strict equality, no layout restrictions

In [ ]:
# Minimal CNOT synthesis example (S)
from qsynth import cnot_synthesis
result = cnot_synthesis(circuit=qc, metric='cx-count')
print('Optimized circuit:')
print(result.circuit)

2. Allowing output qubit permutation (W): weak equality, no layout restrictions

    The result can be further reduced by allowing a permutation on the output qubits:

In [ ]:
# CNOT synthesis with output qubit permutation (W)
from qsynth import cnot_synthesis
result = cnot_synthesis(circuit=qc, metric='cx-count', output_qubit_permute=True)
print('Optimized circuit:')
print(result.circuit)
print('Final mapping:', result.final_mapping)

### Layout-aware CNOT synthesis

3. Layout aware synthesis (S+R): strict equality, layout restrictions

    Layout restrictions can be specified by just providing a `coupling_graph`.
    Here we define the nearest neighbour platform on 4 qubits as `line-4`.

In [ ]:
# Layout aware CNOT synthesis (S+R)
from qsynth import cnot_synthesis, get_coupling_graph
coupling_graph = get_coupling_graph("line-4")
print("Coupling graph: ", coupling_graph)
result = cnot_synthesis(circuit=qc, metric='cx-count', coupling_graph=coupling_graph)
print('Optimized circuit:')
print(result.circuit)

4. Layout aware with output qubit permutation (W+R): weak equivalence, with layout restrictions

    Again, the result can be reduced by allowing a permutation on the output qubits.

In [ ]:
# Layout aware CNOT synthesis with output qubit permutation (W+R)
from qsynth import cnot_synthesis

result = cnot_synthesis(circuit=qc, metric='cx-count', coupling_graph=coupling_graph, output_qubit_permute=True)
print('Optimized circuit:')
print(result.circuit)
print('Final mapping:', result.final_mapping)

### Optimization metrics

CNOT synthesis supports optimizing for the number of CNOT gates (`cx-count`) or the depth of CNOT gates (`cx-depth`).
Additionally, we allow *bounding* a metric to the current value.
For instance we can optimize `cx-depth` and bound `cx-count` to get the minimal CNOT depth without increasing the CNOT count.
Here we resynthesize the circuit from above but reduce the CNOT depth to 4 while keeping the CNOT count of 5:

In [ ]:
result = cnot_synthesis(circuit=result.circuit, metric='cx-depth', bound_metric='cx-count', coupling_graph=coupling_graph, output_qubit_permute=True, verbose=0)
print('Optimized circuit:')
print(result.circuit)

## CNOT+Rz Synthesis

CNOT+Rz synthesis is an extension of CNOT synthesis, allowing {Rz, Z, T, S, Tdg, Sdg} gates alongside CNOT and SWAP gates.
All synthesis variants of CNOT synthesis are also available for CNOT+Rz synthesis (S, W, S+R, W+R).
We also support the same metrics (`cx-count` and `cx-depth`) and the possibility of bounding these metrics.
CNOT+Rz synthesis of arbitrary quantum circuits is available through the `cnot_rz_peephole_synthesis` function (see [Peephole Synthesis](#peephole-synthesis) below).
Here is a small example of CNOT+Rz synthesis:

In [ ]:
from qsynth import cnot_rz_synthesis

qc = qc_example()
print("Original circuit:")
print(qc)
result = cnot_rz_synthesis(qc, metric="cx-depth", output_qubit_permute=False)
print('Optimized circuit:')
print(result.circuit)

## Clifford Synthesis

Clifford Synthesis can be applied to pure Clifford circuits, recognizing e.g. gates X, Y, Z, H, S, CX, CZ.  
(See also Peephole synthesis for Clifford slices below).

In [ ]:
# Example Clifford Circuit
from qiskit import QuantumCircuit
clifford = QuantumCircuit.from_qasm_file('04q_33936_clifford.qasm')
print('Original circuit:')
print(clifford.draw("text", fold=-1))


Q-Synth can optimize Clifford circuits for `cx-count` and `cx-depth` with SAT while allowing bound metrics as explained in [Optimization metrics](#optimization-metrics).
Using planning, we allow optimizing for `cx-count` or `cx-count_1q-count` (minimizing 1-qubit gates while guaranteeing minimal number of CNOT gates).
We support three variants: S, W, S+R.

1. Minimal example (S): strict equivalence, no layout restrictions  
    (minimizing for `cx-depth`)

In [ ]:
# Minimal Clifford synthesis example (S)
from qsynth import clifford_synthesis
depth_optimal_result = clifford_synthesis(circuit=clifford, metric='cx-depth', verbose=0)
print('Optimized circuit:')
print(depth_optimal_result.circuit.draw("text", fold=-1))

This CNOT-depth optimization has reduced `cx-depth` from 7 to 4 and `cx-count` from 8 to 7.
To further optimize the `cx-count` without increasing `cx-depth`, we can specify `cx-depth` as the `bound_metric`:

In [ ]:
# Clifford synthesis example (S) with combined metric
from qsynth import clifford_synthesis
result = clifford_synthesis(circuit=depth_optimal_result.circuit, metric='cx-count', bound_metric='cx-depth', verbose=0)
print('Optimized circuit:')
print(result.circuit.draw("text", fold=-1))

Note that the `cx-depth` is still 4 but the `cx-count` has been further reduced from 7 to 6.

2. Allowing output qubit permutation (W): weak equivalence

In [ ]:
result = clifford_synthesis(circuit=depth_optimal_result.circuit, metric='cx-count', bound_metric='cx-depth',
                            output_qubit_permute=True, verbose=0)
print('Optimized circuit:')
print(result.circuit.draw("text", fold=-1))
print('Final mapping:', result.final_mapping)

With output qubit permutations, we can reduce this even further to `cx-depth`=3 and `cx-count`=5.

3. Layout aware synthesis (S+R): strict equivalence, with layout restrictions

In [ ]:
from qsynth import get_coupling_graph
coupling_graph = get_coupling_graph(platform="line-4")
result = clifford_synthesis(circuit=clifford, metric='cx-count', coupling_graph=coupling_graph, verbose=0)
print('Optimized circuit:')
print(result.circuit.draw("text", fold=-1))

Now we need 8 CNOTs due to the layout restrictions.

We can reduce single qubit gates in post-processing without changing the optimal CNOT structure.

In [ ]:
from qsynth import clifford_synthesis

def gate_counts(c):
    ops = c.count_ops()
    return ops.get('cx', 0), sum(n for g, n in ops.items() if g != 'cx')

# SAT minimizes the CNOT count only (no single-qubit post-processing)
sat = clifford_synthesis(circuit=clifford, metric='cx-count', postprocess_1q_gates=None, verbose=-1)
# Greedy post-processing: same CNOTs, fewer single-qubit gates
rigid = clifford_synthesis(circuit=clifford, metric='cx-count', postprocess_1q_gates='greedy', verbose=-1)

print('input        :  cx=%d  1q=%d' % gate_counts(clifford))
print('SAT cx-count :  cx=%d  1q=%d' % gate_counts(sat.circuit))
print('SAT + greedy :  cx=%d  1q=%d' % gate_counts(rigid.circuit))

Greedy post-processing keeps the optimal 6 CNOTs but reduces the single-qubit gates from 25 to 13. While effective, planning-based resynthesis may be needed for further reduction of single-qubit gates. For planning-based resynthesis, we use a classical planner, for example the Fast Downward planner with the merge-and-shrink heuristic. For planning-based functionality, please install the Fast Downward planner (more instructions available in [INSTALL_CLI.md](INSTALL_CLI.md)).

To optimize cx-count *and* single-qubit gate count together, we can use **cost-based** planning encoding, this gives strictly smaller circuits than SAT. Since it is quite slow, we only illustrate it on a 3-qubit benchmark.

In [ ]:
three_q = QuantumCircuit.from_qasm_file('03q_55125.qasm')

# SAT minimizes CNOTs only: this input is already CNOT-optimal, so single-qubit gates are untouched
sat3 = clifford_synthesis(circuit=three_q, metric='cx-count', postprocess_1q_gates=None, verbose=-1)
# Rigid post-processing: same CNOTs, fewer single-qubit gates
rigid = clifford_synthesis(circuit=three_q, metric='cx-count', postprocess_1q_gates='rigid', verbose=-1)
# Cost-based planning minimizes CNOTs + single-qubit gates jointly (uses the fd-ms planner)
cost = clifford_synthesis(circuit=three_q, metric='cx-count_1q-count', postprocess_1q_gates=None, model='planning',
                          solver='fd-ms', verbose=-1)

print('input               :  cx=%d  1q=%d' % gate_counts(three_q))
print('SAT cx-count        :  cx=%d  1q=%d' % gate_counts(sat3.circuit))
print('planning rigid-based:  cx=%d  1q=%d' % gate_counts(rigid.circuit))
print('planning cost-based :  cx=%d  1q=%d' % gate_counts(cost.circuit))
print(cost.circuit.draw("text", fold=-1))

The above example shows the usage of stronger planning-based reduction in single qubit gates. First, SAT-based encoding only optimizes the cx-count (already optimal). Using "rigid" post-processing based on planning further reduces the single qubit gates (25 -> 12) without changing the CNOT count. Finally, cost-based encoding produces a circuit with optimal cx-count and 1q-count (6 CNOTs and 8 single qubit gates).

## Peephole Synthesis

Q-Synth can also be used to resynthesize general quantum circuits by slicing them into Clifford or CNOT slices, and resynthesizing each slice optimally.  

In [ ]:
from qiskit import QuantumCircuit
print('Original circuit:')
qc = qc_example()
print(qc)

### CNOT slicing 

Note that the circuit above has 2 CNOT slices, which cannot be optimized:

In [ ]:
from qsynth import cnot_peephole_synthesis

result = cnot_peephole_synthesis(circuit=qc, metric='cx-count')
print('Optimized circuit:')
print(result.circuit)

### Clifford slicing 

Note, the circuit above has a single Clifford slice, so now there is some room for optimization:

In [ ]:
from qsynth import clifford_peephole_synthesis

result = clifford_peephole_synthesis(circuit=qc, metric='cx-count')
print('Optimized circuit:')
print(result.circuit)

### Resynthesizing general quantum circuits
  
We now illustrate the re-synthesis of a non-Clifford quantum circuit consisting of multiple slices, without layout restrictions:

In [ ]:
barenco = QuantumCircuit.from_qasm_file('barenco_tof_3.qasm')
result = clifford_peephole_synthesis(circuit=barenco, metric='cx-count', verbose=0)

For layout-aware Peephole Resynthesis, we expect the input circuit to be already layout mapped.
If it is not layout mapped, one could either use Q-Synth or Qiskit to first perform layout mapping and then do peephole re-synthesis.

Here is an example of layout-aware peephole synthesis:

In [ ]:
melbourne = get_coupling_graph(platform="melbourne")
mapped_result = layout_synthesis(circuit=barenco, coupling_graph=melbourne, metric='cx-count', verbose=0)
resynthesized_result = clifford_peephole_synthesis(circuit=mapped_result.circuit, metric='cx-count',
                                                   coupling_graph=melbourne, verbose=0)

Optimal layout synthesis adds 7 swaps, resulting in total 45 CNOTs.
Applying layout-aware peephole Resynthesis reduces the cx-count 45 -> 42 and cx-depth 43 -> 39.

We can reduce the CNOT count and depth even more using CNOT+Rz resynthesis:

In [ ]:
from qsynth import cnot_rz_peephole_synthesis

cnot_rz_result = cnot_rz_peephole_synthesis(resynthesized_result.circuit, metric='cx-count', verbose=0)